🧹**Data Preprocessing for Transformers Models**

-> URLs: URLs are replaced with the term url.

-> User Mentions: Mentions (e.g., @username) are replaced with the term user.

-> Hashtags: Hashtags (e.g., #hashtag) are removed, keeping only the text.

-> Extra Spaces: Multiple spaces are collapsed into a single space.

-> Digits: Any numbers in the text are removed.

-> Convert Emojis and Slang: Emojis and internet slang abbreviations are converted into meaningful text using custom functions.

In [1]:
from Lists.abbreviations import INTERNET_SLANG
from Lists.emoticons import EMOTICONS
from Lists.emojis import EMO_UNICODE
import pandas as pd
import contractions
import unicodedata
import re

As part of the preprocessing, we remove all rows where the `cyberbullying_type` is labeled as `not_cyberbullying`. This category tends to be too broad and ambiguous, and it has been observed to significantly reduce model performance. By excluding these samples, we aim to improve the consistency and accuracy of our classification models.


In [2]:
df = pd.read_csv('./cyberbullying/cyberbullying_tweets_translated.csv')
df=df.dropna(how='any')
df = df.drop(columns=['label_ID'])

df = df.rename(columns={
    'tweet_text': 'text',
    'cyberbullying_type': 'type'
})

df = df[df["type"]!="not_cyberbullying"]

In [3]:
type_labels = {label: idx for idx, label in enumerate(df['type'].unique())}
print(type_labels)
df['label'] = df['type'].map(type_labels)

{'gender': 0, 'religion': 1, 'other_cyberbullying': 2, 'age': 3, 'ethnicity': 4}


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39739 entries, 7945 to 47683
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    39739 non-null  object
 1   type    39739 non-null  object
 2   label   39739 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [6]:
df.head()

,text,type,label
7945,rape is real..zvasiyana nema jokes about being...,gender,0
7946,You never saw any celebrity say anything like ...,gender,0
7947,"@ManhattaKnight I mean he's gay, but he uses g...",gender,0
7948,RT @Raul_Novoa16: @AliciaBernardez @Alex_Aim @...,gender,0
7949,Rape is rape. And the fact that I read one pos...,gender,0


In [7]:
UNICODE_EMO = {v: k for k, v in EMO_UNICODE.items()}


def convert_emoticons(text):
    for emot in EMOTICONS:
        text = re.sub(u'(' + emot + ')', "  ".join(EMOTICONS[emot].replace(",", "").split()), text)
    return text


def convert_emojis(text):
    for emot in UNICODE_EMO:
        text = re.sub(r'(' + emot + ')', "  ".join(UNICODE_EMO[emot].replace(",", "").replace(":", "").split()), text)
    return text


def convert_abbreviations(text):
    for abbr in INTERNET_SLANG:
        pattern = re.compile(r'\b' + re.escape(abbr) + r'\b', re.IGNORECASE)
        replacement = " ".join(INTERNET_SLANG[abbr].replace(",", "").split())
        text = pattern.sub(replacement, text)
    return text


def remove_unknown_emojis(text):
    emoji_pattern = re.compile("["
                               u"\U0001F600-\U0001F64F"  # emoticons
                               u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                               u"\U0001F680-\U0001F6FF"  # transport & map symbols
                               u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                               u"\U00002702-\U000027B0"  # miscellaneous symbols
                               u"\U0001F900-\U0001F9FF"  # supplemental symbols
                               u"\U0001F200-\U0001F251"  # enclosed characters
                               u"\U0001F004-\U0001F0CF"  # playing cards
                               u"\U00002B50"  # star symbol
                               "]+", flags=re.UNICODE)

    emojis_in_text = emoji_pattern.findall(text)

    for emoji_char in emojis_in_text:
        if emoji_char not in UNICODE_EMO:
            text = text.replace(emoji_char, '')
    return text


def remove_accented_chars(text):
    text = contractions.fix(text)
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8', 'ignore')

In [8]:
def convert_emojis_and_slang(text):
    text = convert_emoticons(text)
    text = convert_emojis(text)
    text = remove_unknown_emojis(text)
    text = convert_abbreviations(text)
    text=remove_accented_chars(text)

    return text

In [9]:
from tqdm.notebook import tqdm

In [10]:
def preprocess_text(df_input):
    tqdm.pandas() 

    df_comments = df_input.copy()
    df_comments['text'] = df_comments['text'].str.replace(r"http[s]?://\S+", "URL", regex=True)  # remove URLs
    df_comments['text'] = df_comments['text'].str.replace(r'@[A-Za-z0-9_.]+', 'USER', regex=True)  # remove user mentions
    df_comments['text'] = df_comments['text'].str.replace(r'#+(\S+)', r'\1', regex=True)  # remove hashtags
    df_comments['text'] = df_comments['text'].str.replace(r'\s+', ' ', regex=True)  # remove extra spaces
    df_comments['text'] = df_comments['text'].str.replace(r'\d+', '', regex=True)  # remove digits

    df_comments['text'] = df_comments['text'].progress_apply(lambda x: convert_emojis_and_slang(x))

    df_comments = df_comments.dropna(how='any')
    df_comments = df_comments.drop_duplicates()

    return df_comments


In [11]:
df = preprocess_text(df)

  0%|          | 0/39739 [00:00<?, ?it/s]

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 39290 entries, 7945 to 47683
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    39290 non-null  object
 1   type    39290 non-null  object
 2   label   39290 non-null  int64 
dtypes: int64(1), object(2)
memory usage: 1.2+ MB


In [13]:
df.to_csv('./cyberbullying_tweets_transformers.csv',index=False)